# Cleaning Messy Geotechnical Soil Data

An end-to-end data cleaning pipeline in Python (pandas, seaborn) for a messy borehole soil dataset, built after completing the Kaggle Data Cleaning course.

**Steps:** fix data types, standardize categories, remove duplicates, handle missing values, cap outliers with the 1.5 × IQR rule, and verify with boxplots.

*Requires `soil_data.csv` in the same folder.*

In [91]:
#Step1:Import libraries and Load the dataset.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
soil_data=pd.read_csv('soil_data.csv')
soil_data

,Unnamed: 0,sample_id,site,soil_type,depth_m,moisture_pct,dry_density_kgm3,liquid_limit,cohesion_kpa,test_date
0,0,SM-0490,yaba,Silt,2.0,10.4,1773.0,55.7,23.0,2025-04-04
1,1,SM-0430,Yaba,Silt,2.9,25.2,1747.0,36.4,56.0,2025-05-29
2,2,SM-0083,IKEJA,Sand,5.1,14.1,1661.0,68.2,NaN,2025-06-19
3,3,SM-0121,Ikoyi,Sand,9.1,24.1,1769.0,40.6,NaN,2025-10-09
4,4,SM-0222,Ikoyi,Silt,0.8,NaN,1893.0,36.4,19.0,2025-07-31
...,...,...,...,...,...,...,...,...,...,...
510,510,SM-0212,Yaba,Clay,5.5,17.4,1557.0,55.2,15.3,2025-05-01
511,511,SM-0068,Lekki,Sand,6.7,11.1,1711.0,45.1,NaN,2025-06-13
512,512,SM-0048,Ajah,Laterite,2.4,22.5,1607.0,37.9,8.7,2025-04-28
513,513,SM-0197,Lekki,Laterite,8.7,19.9,1606.0,36.9,30.0,2025-06-03


In [92]:
#Step2:Inspect the dataset (use shape,info(),describe())
soil_data.shape #returns the number of rows and columns of the dataset
soil_data.info() #returns the Non-null count and dtype of each column
soil_data.describe() #gives a statistical summary for the numerical columns of the dataset.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 515 entries, 0 to 514
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        515 non-null    int64  
 1   sample_id         515 non-null    object 
 2   site              515 non-null    object 
 3   soil_type         507 non-null    object 
 4   depth_m           515 non-null    float64
 5   moisture_pct      480 non-null    float64
 6   dry_density_kgm3  515 non-null    float64
 7   liquid_limit      500 non-null    float64
 8   cohesion_kpa      424 non-null    float64
 9   test_date         515 non-null    object 
dtypes: float64(5), int64(1), object(4)
memory usage: 40.4+ KB


,Unnamed: 0,depth_m,moisture_pct,dry_density_kgm3,liquid_limit,cohesion_kpa
count,515.000000,515.000000,480.000000,515.000000,500.000000,424.000000
mean,257.000000,6.357087,21.738958,1687.720388,45.487600,28.374528
std,148.811962,5.167122,6.034539,128.431648,10.264526,22.753726
min,0.000000,0.500000,7.200000,400.000000,17.400000,1.500000
25%,128.500000,3.250000,18.400000,1608.000000,39.200000,13.675000
50%,257.000000,6.000000,21.900000,1699.000000,45.200000,26.500000
75%,385.500000,9.150000,25.100000,1767.500000,51.950000,38.500000
max,514.000000,95.000000,85.000000,2043.000000,140.000000,310.000000


In [78]:
soil_data.columns


Index(['Unnamed: 0', 'sample_id', 'site', 'soil_type', 'depth_m',
       'moisture_pct', 'dry_density_kgm3', 'liquid_limit', 'cohesion_kpa',
       'test_date'],
      dtype='object')

In [60]:
#Step3:Convert test_date columns to  datetime64[ns]
soil_data['test_date']=pd.to_datetime(soil_data['test_date'])
soil_data.dtypes

Unnamed: 0                   int64
sample_id                   object
site                        object
soil_type                   object
depth_m                    float64
moisture_pct               float64
dry_density_kgm3           float64
liquid_limit               float64
cohesion_kpa               float64
test_date           datetime64[ns]
dtype: object

In [71]:
#Step 4:Convert the soil_type column's data to category:
soil_data['soil_type']=soil_data['soil_type'].astype('category')
soil_data.dtypes

Unnamed: 0             int64
sample_id             object
site                  object
soil_type           category
depth_m              float64
moisture_pct         float64
dry_density_kgm3     float64
liquid_limit         float64
cohesion_kpa         float64
test_date             object
dtype: object

In [26]:
#Step 5:Check for inconsistent data(spellings) in columns:
soil_data['site'].unique()
#this indicates inconsistent spellings in the 'site' column

array(['yaba', 'Yaba', '  IKEJA ', 'Ikoyi', 'lekki', 'Ikeja', 'ikeja',
       'Lekki', 'Ajah', 'ajah', 'ikoyi', '  LEKKI ', '  IKOYI ',
       '  YABA ', '  AJAH '], dtype=object)

In [29]:
#Correction of inconsistent data:
soil_data['site']=soil_data['site'].str.strip().str.title()
soil_data['site'].unique() 
#all inconsistent data in the site columns have been corrected.

array(['Yaba', 'Ikeja', 'Ikoyi', 'Lekki', 'Ajah'], dtype=object)

In [33]:
#Step 6:Checking for duplicate rows deleting them:
soil_data.duplicated().sum() 

0

In [49]:
#Step 7:Check for missing values and correct them:
soil_data.isnull().sum()
(soil_data.isnull().mean()*100).round(2)

Unnamed: 0           0.00
sample_id            0.00
site                 0.00
soil_type            0.00
depth_m              0.00
moisture_pct         6.80
dry_density_kgm3     0.00
liquid_limit         2.91
cohesion_kpa        17.67
test_date            0.00
dtype: float64

In [73]:
soil_data['soil_type']=soil_data['soil_type'].fillna('Unknown')
#all missing values of the 'soil_type' column have been replaced by 'unknown'
soil_data['moisture_pct']=soil_data['moisture_pct'].fillna(soil_data['moisture_pct'].median())
soil_data['liquid_limit']=soil_data['liquid_limit'].fillna(soil_data['liquid_limit'].median())
soil_data['cohesion_kpa']=soil_data['cohesion_kpa'].fillna(soil_data['cohesion_kpa'].median())
soil_data.head()

,Unnamed: 0,sample_id,site,soil_type,depth_m,moisture_pct,dry_density_kgm3,liquid_limit,cohesion_kpa,test_date
0,0,SM-0490,yaba,Silt,2.0,10.4,1773.0,55.7,23.0,2025-04-04
1,1,SM-0430,Yaba,Silt,2.9,25.2,1747.0,36.4,56.0,2025-05-29
2,2,SM-0083,IKEJA,Sand,5.1,14.1,1661.0,68.2,26.5,2025-06-19
3,3,SM-0121,Ikoyi,Sand,9.1,24.1,1769.0,40.6,26.5,2025-10-09
4,4,SM-0222,Ikoyi,Silt,0.8,21.9,1893.0,36.4,19.0,2025-07-31


In [107]:
#Step 8:Detection and Capping of Outliers for numerical columns
num_cols=soil_data[['depth_m','moisture_pct','dry_density_kgm3','liquid_limit','cohesion_kpa']]
for col in num_cols:
    Q1=num_cols[col].quantile(0.25)
    Q3=num_cols[col].quantile(0.75)
    IQR=Q3-Q1
    lower_limit=Q1-1.5*IQR
    upper_limit=Q3+1.5*IQR
    num_cols[col].clip(lower=lower_limit,upper=upper_limit)
soil_data[['depth_m','moisture_pct','dry_density_kgm3','liquid_limit','cohesion_kpa']].replace(num_cols)
#all outliers have been replaced by the lower and upper limits of each column

,depth_m,moisture_pct,dry_density_kgm3,liquid_limit,cohesion_kpa
0,5.1,10.4,1773.0,55.7,NaN
1,2.9,25.2,1747.0,36.4,42.0
2,5.1,14.1,1661.0,68.2,NaN
3,9.1,24.1,1769.0,40.6,NaN
4,0.8,NaN,1893.0,36.4,20.8
...,...,...,...,...,...
510,5.5,17.4,1557.0,55.2,15.3
511,6.7,11.1,1711.0,45.1,NaN
512,2.4,22.5,1607.0,37.9,8.7
513,8.7,19.9,1606.0,36.9,NaN
